# 🟢 Solution: NTK-aware RoPE Scaling

Reference solution for NTK-aware Rotary Position Embedding scaling.

$$\text{base}_{\text{new}} = 10000 \cdot \text{scale}^{D/(D-2)}$$

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def ntk_rope(q, k, scale):
    """
    NTK-aware RoPE 缩放，用于长上下文外推。

    标准 RoPE 在超过训练上下文长度时性能下降，因为高频维度的旋转角度超出训练分布。
    NTK-aware 缩放通过调整基底频率 (base) 来拉伸低频维度，同时保留高频维度，
    从而在不微调的情况下支持更长的上下文。

    参数:
        q, k: 查询和键张量, shape = (B, S, D)
            B = batch size, S = 序列长度, D = head 维度（必须为偶数）
        scale: 上下文长度比例，如 4.0 表示支持 4 倍于训练长度的上下文

    返回: 旋转后的 (q, k)，形状不变
    """
    B, S, D = q.shape

    # ---- Step 1: 计算新的基底频率 ----
    # 标准 RoPE 的 base = 10000
    # NTK-aware 缩放: base_new = 10000 * scale^(D/(D-2))
    # 为什么是 D/(D-2)？这是为了让缩放后的频率分布恰好「填补」新增的上下文长度
    # scale 越大，base 越大，频率越低，旋转越慢 → 支持更长序列
    new_base = 10000.0 * (scale ** (D / (D - 2)))

    # ---- Step 2: 计算各维度的旋转频率 ----
    # pos: 位置索引 [0, 1, ..., S-1], shape: (S, 1)
    # dim: 维度索引 [0, 2, 4, ..., D-2], shape: (D/2,)
    # freqs[i] = 1 / base^(dim[i]/D)
    # 低维度 (dim 小) → 频率高 → 旋转快（捕捉局部位置信息）
    # 高维度 (dim 大) → 频率低 → 旋转慢（捕捉全局位置信息）
    pos = torch.arange(S, device=q.device).float().unsqueeze(1)  # (S, 1)
    dim = torch.arange(0, D, 2, device=q.device).float()          # (D/2,)
    freqs = 1.0 / (new_base ** (dim / D))                         # (D/2,)

    # ---- Step 3: 计算旋转角度 ----
    # angles[s, d] = pos[s] * freqs[d]
    # shape: (S, 1) * (D/2,) = (S, D/2)  通过广播
    angles = pos * freqs  # (S, D/2)
    cos_a = torch.cos(angles)  # (S, D/2)
    sin_a = torch.sin(angles)  # (S, D/2)

    # ---- Step 4: 旋转函数 ----
    # RoPE 的旋转操作: 将每对相邻维度 (x1, x2) 视为 2D 向量，旋转 angle 角度
    # 旋转矩阵: [[cos, -sin], [sin, cos]]
    # x1' = x1 * cos - x2 * sin
    # x2' = x1 * sin + x2 * cos
    # x[..., 0::2] 取偶数维 (x1), x[..., 1::2] 取奇数维 (x2)
    # cos_a, sin_a: (S, D/2) 通过广播与 x1, x2 的 (B, S, D/2) 对齐
    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]  # 各 (B, S, D/2)
        # stack 在最后一维拼接: (B, S, D/2, 2)
        # flatten(-2) 把最后两维合并: (B, S, D)
        return torch.stack([x1 * cos_a - x2 * sin_a,
                            x1 * sin_a + x2 * cos_a], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

In [ ]:
# Verify: compare scale=1 vs scale=4 frequencies, show norm preservation
B, S, D = 1, 8, 16
q = torch.randn(B, S, D)
k = torch.randn(B, S, D)

q1, k1 = ntk_rope(q, k, scale=1.0)
q4, k4 = ntk_rope(q, k, scale=4.0)

print("scale=1 base:", 10000.0 * (1.0 ** (D / (D - 2))))
print("scale=4 base:", 10000.0 * (4.0 ** (D / (D - 2))))
print()
print("Norm preservation (scale=1):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q1.norm(dim=-1).mean().item())
print()
print("Norm preservation (scale=4):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q4.norm(dim=-1).mean().item())

In [ ]:
from torch_judge import check
check("ntk_rope")

## 📝 核心思路总结

1. **RoPE 的本质**：把每对相邻维度当作 2D 向量做旋转，旋转角度由位置和频率共同决定，从而编码相对位置信息。
2. **NTK-aware 缩放只改 base**：`base_new = 10000 * scale^(D/(D-2))`，不改变旋转公式本身，通过调低频率来支持更长上下文。
3. **频率的多尺度特性**：低维度频率高（旋转快，编码局部位置），高维度频率低（旋转慢，编码全局位置），NTK 缩放主要拉伸低频维度。
4. **旋转是正交变换**：`[x1*cos - x2*sin, x1*sin + x2*cos]` 保持向量范数不变，这是 RoPE 能保持注意力分数相对性的关键。